In [1]:
from config import *
from functions.useful_functions import *            #useful functions, defined in the functions/useful_functions.py file
from functions.distributions_and_correlations import *
from functions.galaxy_distribution import *

import sys
import os
from collections import defaultdict

load_correlations(filename="correlations")

In [2]:
import json


In [23]:
print(maxsamp)

1000000


In [51]:
nsamples_pm = load_file("nsample_dicts/LLLe/scovpm")

print(nsamples_pm)

defaultdict(<class 'list'>, {'0': {'00': 3000000, '01': 200000, '02': 2000, '03': 500, '04': 300, '10': 5000000, '11': 60000, '12': 2000, '13': 5000, '14': 90000, '20': 500000000, '21': 500000, '22': 800000, '23': 2000, '24': 2000, '30': 200000000, '31': 5000000, '32': 200000, '33': 1000000, '34': 2000, '40': 60000000, '41': 10000000, '42': 1000000, '43': 100000, '44': 200000}, '1': {'00': 3000000, '01': 200000, '02': 2000, '03': 500, '04': 300, '10': 6000000, '11': 50000, '12': 2000, '13': 4000, '14': 70000, '20': 60000000, '21': 400000, '22': 600000, '23': 2000, '24': 2000, '30': 200000000, '31': 2000000, '32': 200000, '33': 1000000, '34': 2000, '40': 50000000000, '41': 6000000, '42': 500000, '43': 90000, '44': 200000}, '2': {'00': 3000000, '01': 200000, '02': 2000, '03': 500, '04': 300, '10': 4000000, '11': 50000, '12': 2000, '13': 4000, '14': 60000, '20': 30000000, '21': 300000, '22': 900000, '23': 2000, '24': 2000, '30': 9000000000, '31': 1000000, '32': 100000, '33': 600000, '34':

In [16]:
def convert_to_sci_notation(d):
    """ Recursively convert numbers to scientific notation strings. """
    if isinstance(d, dict):
        return {k: convert_to_sci_notation(v) for k, v in d.items()}
    elif isinstance(d, (int, float)):
        return f"{d:.0e}"  # Convert to scientific notation as a string
    return d  # Keep everything else unchanged

# Convert before dumping
formatted_dict = convert_to_sci_notation(nsamples_pp)

# Dump to JSON
print(json.dumps(formatted_dict, indent=4))


{
    "00": {
        "00": "1e+03",
        "01": "6e+02",
        "02": "4e+02",
        "03": "3e+02",
        "04": "3e+02",
        "10": "6e+02",
        "11": "1e+03",
        "12": "6e+02",
        "13": "4e+02",
        "14": "3e+02",
        "20": "4e+02",
        "21": "6e+02",
        "22": "2e+03",
        "23": "9e+02",
        "24": "6e+02",
        "30": "3e+02",
        "31": "4e+02",
        "32": "9e+02",
        "33": "3e+03",
        "34": "1e+03",
        "40": "3e+02",
        "41": "3e+02",
        "42": "6e+02",
        "43": "1e+03",
        "44": "3e+03"
    },
    "01": {
        "00": "1e+03",
        "01": "6e+02",
        "02": "4e+02",
        "03": "3e+02",
        "04": "3e+02",
        "10": "6e+02",
        "11": "1e+03",
        "12": "6e+02",
        "13": "4e+02",
        "14": "3e+02",
        "20": "4e+02",
        "21": "7e+02",
        "22": "2e+03",
        "23": "9e+02",
        "24": "6e+02",
        "30": "3e+02",
        "31": "4e+02",
  

In [4]:
distribution_LL = Distributions(Nlens, binscheme=binscheme_LL, Nbina=Nbina_LL, Thetamax=Thetamax_LL) 
distribution_Le = Distributions(NGal, binscheme=binscheme_Le, Nbina=Nbina_Le, Thetamax=Thetamax_Le)
distribution_Lp = Distributions(NGal, binscheme=binscheme_Lp, Nbina=Nbina_Lp, Thetamax=Thetamax_Lp)

distributions = {"LL": distribution_LL,
                "Le": distribution_Le,
                "Lp": distribution_Lp}


In [5]:
get_item('xi2_LOS_plus_intp','xi1_LOS_plus_intp','xi32_LOS_plus_intp','xi2_LOS_minus_intp','xi1_LOS_minus_intp','xi32_LOS_minus_intp')
get_item('xi2_eps_plus_intp','xi1_eps_plus_intp','xi2_eps_minus_intp','xi1_eps_minus_intp')
get_item('xi2_d_intp')
get_item('xi2_LOS_eps_plus_intp', 'xi2_LOS_eps_minus_intp', 'xi32_LOS_eps2_plus_intp', 'xi32_LOS_eps2_minus_intp', 'xi32_LOS2_eps_plus_intp', 'xi32_LOS2_eps_minus_intp')
get_item('xi2_LOS_d_intp')
get_item('xi2_d_eps_intp')

In [42]:
errors = []
time_dictionary = {}
time_dictionary = defaultdict(list)

def generate_ncov_LeLe(distributions, sigma_noise, sigma_shape, b1, b2, approx=False):
    time_dictionary[str(b1)+str(b2)] = {}
    """
    Computes the contribution of noise in the covariance matrix
    of the LOS shear - ellipticity correlation functions.

    distributions  : statistics relating to the distributions of lenses and galaxies
    sigma_noise    : the noise on the lens measurements
    sigma_shape    : the intrinsic galaxy shapes
    b1             : the galaxy redshift bin b  (0 to 4)
    b2             : the galaxy redshift bin b' (0 to 4)
    approx         : Bool, True to give the approximate result (quicker)
    """

    distribution = distributions['Le']
    
    Omegatot   = distribution.Omegatot   #\Omega in the math - the total solid angle covered by the survey
    Nbin       = distribution.Nbina      #the number of angular bins
    Omegas     = distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2)
    rs         = distribution.limits     #the angular bin limits (in rad)
    
    Ngal1    = get_ngal(b1)        #G_b in the math - the number of galaxies in redshift bin b
    Ngal2    = get_ngal(b2)        #G_{b'} in the math - the number of galaxies in redshift bin b'
    
    # Initialise the tables with their non-integral diagonal elements

    if b1 == b2:
    
        diagonal = ( Omegatot / ( 2 * Nlens * Ngal1 * Omegas) ) * (sigma_noise**2 * sigma_shape**2 
                       +xi1_LOS_plus_intp(0)*sigma_shape**2 + xi1_eps_plus_intp[b1](0)*sigma_noise**2)

    else:
        
        diagonal = np.zeros(Nbin)
    
    ncov_pp = np.diag(diagonal)
    ncov_mm = np.diag(diagonal)

    #note that, like in the autocorrelations case, the pm and mp covariance matrices have no diagonal terms
    ncov_pm = np.zeros_like(ncov_pp)
    ncov_mp = np.zeros_like(ncov_pp)
    
    if not approx:
        
        # Define the integrands
        # I'll include everything which isn't constant between the terms as the integrand

        def integrand_pp(params):
            r1, r2, psi2 = params  # Unpack the input array
            r = np.sqrt(r1**2 + r2**2 - 2 * r1 * r2 * np.cos(psi2))           #the equivalent of l-l'

            if b1 == b2:
                f = 2 * np.pi * r1 * r2 * (sigma_shape**2 * xi2_LOS_plus_intp(r)  + sigma_noise**2 * Ngal2 * xi2_eps_plus_intp[b1][b2](r) / Nlens)
            else:
                f = 2 * np.pi * r1 * r2 * ( sigma_noise**2 * Ngal2 * xi2_eps_plus_intp[b1][b2](r) / Nlens)
            
            return f

        def integrand_pm(params):
            r1, r2, psi2 = params  # Unpack the input array
            x = r1 - r2 * np.cos(psi2)
            y = - r2 * np.sin(psi2)
            r = np.sqrt(x**2 + y**2)
            psi = np.arctan2(y, x) 

            if b1 == b2:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * (psi-psi2)) * (sigma_shape**2 * xi2_LOS_minus_intp(r) + sigma_noise**2 * Ngal2 * xi2_eps_minus_intp[b1][b2](r) / Nlens)
            else:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * (psi-psi2)) * (sigma_noise**2 * Ngal2 * xi2_eps_minus_intp[b1][b2](r) / Nlens)
            
            return f

        def integrand_mp(params):
            r1, r2, psi2 = params  # Unpack the input array
            x = r1 - r2 * np.cos(psi2)
            y = - r2 * np.sin(psi2)
            r = np.sqrt(x**2 + y**2)
            psi = np.arctan2(y, x)

            if b1 == b2:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * psi) * (sigma_shape**2 * xi2_LOS_minus_intp(r) + sigma_noise**2 * Ngal2 * xi2_eps_minus_intp[b1][b2](r) / Nlens)
            else:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * psi) * (sigma_noise**2 * Ngal2 * xi2_eps_minus_intp[b1][b2](r) / Nlens)
            
            return f

        def integrand_mm(params):
            r1, r2, psi2 = params  # Unpack the input array
            r = np.sqrt(r1**2 + r2**2 - 2 * r1 * r2 * np.cos(psi2))

            if b1 == b2:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * psi2) * (sigma_shape**2 * xi2_LOS_plus_intp(r)  + sigma_noise**2 * Ngal2 * xi2_eps_plus_intp[b1][b2](r) / Nlens)
            else:
                f = 2 * np.pi * r1 * r2 * np.cos(4 * psi2) * (sigma_noise**2 * Ngal2 * xi2_eps_plus_intp[b1][b2](r) / Nlens)
                
            return f
            
        def integral_term(integrand, a1, a2):
            integral, err = monte_carlo_integrate(integrand, [(rs[a1], rs[a1+1]), (rs[a2], rs[a2+1]), (0, 2*np.pi)])
            time_dictionary[str(b1)+str(b2)][str(a1)+str(a2)] = roundsf((((err / integral)/0.1) ** 2) * nsamp)
            return integral
        
        # Compute and add the integral contribution
        
        for a1 in range(Nbin):

            for a2 in range(Nbin): # while pm isn't

                # ncov_pm[a1, a2] += integral_term(integrand_pm, a1, a2)
                ncov_mp[a1, a2] += integral_term(integrand_mp, a1, a2)
            
    
    # Make the full noise covariance matrix, with first xi_+ and then xi_-
    ncov = np.block([[ncov_pp, ncov_pm],
                     [ncov_mp, ncov_mm]])
    
    
    return ncov, ncov_pp, ncov_mm, ncov_pm, ncov_mp


In [43]:
for b1 in range(5):
    for b2 in range(5):
        x = generate_ncov_LeLe(distributions, sigma_noise, sigma_shape, b1, b2, approx=False)

In [44]:
max_value = max(
    value for subdict in time_dictionary.values() for value in subdict.values()
)
print(f"{max_value:.0e}")

2e+10


In [45]:
save_pickle(time_dictionary, 'nsample_dicts/LeLe/ncovmp', 'timing')

Successfully saved timing at nsample_dicts/LeLe/ncovmp


In [40]:
def convert_to_sci_notation(d):
    """ Recursively convert numbers to scientific notation strings. """
    if isinstance(d, dict):
        return {k: convert_to_sci_notation(v) for k, v in d.items()}
    elif isinstance(d, (int, float)):
        return f"{d:.0e}"  # Convert to scientific notation as a string
    return d  # Keep everything else unchanged

# Convert before dumping
formatted_dict = convert_to_sci_notation(time_dictionary)

# Dump to JSON
print(json.dumps(formatted_dict, indent=4))


{
    "00": {
        "00": "9e+06",
        "01": "2e+05",
        "02": "2e+03",
        "03": "5e+02",
        "04": "3e+02",
        "10": "3e+06",
        "11": "6e+04",
        "12": "2e+03",
        "13": "5e+03",
        "14": "1e+05",
        "20": "5e+07",
        "21": "4e+05",
        "22": "1e+06",
        "23": "2e+03",
        "24": "2e+03",
        "30": "2e+08",
        "31": "3e+06",
        "32": "1e+05",
        "33": "5e+05",
        "34": "2e+03",
        "40": "2e+08",
        "41": "7e+06",
        "42": "8e+05",
        "43": "9e+04",
        "44": "1e+05"
    },
    "01": {
        "00": "1e+07",
        "01": "2e+05",
        "02": "2e+03",
        "03": "5e+02",
        "04": "3e+02",
        "10": "3e+06",
        "11": "6e+04",
        "12": "2e+03",
        "13": "5e+03",
        "14": "1e+05",
        "20": "4e+08",
        "21": "6e+05",
        "22": "9e+05",
        "23": "2e+03",
        "24": "2e+03",
        "30": "1e+08",
        "31": "4e+06",
  

In [12]:
print(f"'{a1}' : {roundsf((((err / integral)/0.1) ** 2) * nsamp):.0e}, ")

In [ ]:


        time_dictionary[str(b1)+str(b2)] = {}
            time_dictionary[str(b1)+str(b2)][str(a1)+str(a2)] = roundsf((((err / integral)/0.1) ** 2) * nsamp)

In [98]:
print(f"{max(time_dictionary.values()):.0e}, ")
# print(time_dictionary)
#print(f"'{a1}' : {roundsf((((err / integral)/0.1) ** 2) * nsamp):.0e}, ")

TypeError: unsupported format string passed to dict.__format__

In [ ]:
def process_pair(args):
    """Generates and saves covariance matrices for given parameters."""
    b1, b2, distributions, sigma_noise, sigma_shape, cov_matrix = args

    # Create output directory if it doesn't exist
    output_dir = f"matrices/{cov_matrix}"
    os.makedirs(output_dir, exist_ok=True)

    # Generate binned correlation functions
    correlation_data = generate_binned_correlation(distributions, cov_matrix, b1, b2)

    if correlation_data:
        for suffix, data in correlation_data.items():
            if b1 is not None:
                filename = f"{output_dir}/{suffix}{b1}"
            else:
                filename = f"{output_dir}/{suffix}"
            save_pickle(data, filename, f"{suffix} for b1={b1}, b2={b2}, cov_matrix={cov_matrix}")

    # Generate covariance matrices
    ncov_funcs = globals().get(f"generate_ncov_{cov_matrix}")
    ccov_funcs = globals().get(f"generate_ccov_{cov_matrix}")
    scov_funcs = globals().get(f"generate_scov_{cov_matrix}")

    if not (ncov_funcs and ccov_funcs and scov_funcs):
        print(f"Error: Covariance function(s) not found for {cov_matrix}")
        return

    ncov, ncov_pp, ncov_mm, ncov_pm, ncov_mp = ncov_funcs(distributions, sigma_noise, sigma_shape, b1, b2)
    ccov, ccov_pp, ccov_mm, ccov_pm, ccov_mp = ccov_funcs(distributions, b1, b2)
    scov, scov_pp, scov_mm, scov_pm, scov_mp = scov_funcs(distributions, b1, b2)

    # Store results
    covariance_matrices = {
        "ncov": {'full': ncov, 'pp': ncov_pp, 'mm': ncov_mm, 'pm': ncov_pm, 'mp': ncov_mp},
        "ccov": {'full': ccov, 'pp': ccov_pp, 'mm': ccov_mm, 'pm': ccov_pm, 'mp': ccov_mp},
        "scov": {'full': scov, 'pp': scov_pp, 'mm': scov_mm, 'pm': scov_pm, 'mp': scov_mp}
    }

    print(f"Generated all matrices (b1={b1}, b2={b2}, cov_matrix={cov_matrix})")

    # Save matrices
    for name, data in covariance_matrices.items():
        filename = f"{output_dir}/{name}"
        if b1 is not None:
            filename += f"{b1}"
        if b2 is not None:
            filename += f"{b2}"

        save_pickle(data, filename, f"{name} for b1={b1}, b2={b2}, cov_matrix={cov_matrix}")

########################################### 4.2 Executing the function #######################################################
##############################################################################################################################

print("Running")

def main():
    # Define parameter ranges
    b1_values = [0, 1, 2, 3, 4]
    b2_values = [0, 1, 2, 3, 4]
    
    cov_matrices_full = ['LpLe', 'LeLe', 'LpLp']   # Needs (b1, b2)
    cov_matrices_b1 = ['LLLp', 'LLLe']             # Needs only b1
    cov_matrices_no_b = ['LLLL']                   # No (b1, b2)

    args_list = []

    # Full (b1, b2) iteration
    args_list.extend((b1, b2, distributions, sigma_noise, sigma_shape, cov_matrix)
                 for b1, b2 in product(b1_values, b2_values) for cov_matrix in cov_matrices_full)

    # Single b1 iteration
    args_list.extend((b1, None, distributions, sigma_noise, sigma_shape, cov_matrix)
                     for cov_matrix, b1 in product(cov_matrices_b1, b1_values))

    # No b1, b2 iteration (only one call)
    args_list.extend((None, None, distributions, sigma_noise, sigma_shape, cov_matrix)
                     for cov_matrix in cov_matrices_no_b)

    
    # Run in parallel using multiprocessing
    with Pool(processes=cpu_count()) as pool:
        pool.map(process_pair, args_list)

if __name__ == "__main__":
    main()

In [386]:
#This is the matrix I got for integrand_s12 eta = 1. Seems strange!

{
    "0": {
        "00": "2e+05",
        "01": "3e-02",
        "02": "1e-07",
        "03": "1e-08",
        "04": "3e-07",
        "10": "2e+05",
        "11": "3e-02",
        "12": "1e-07",
        "13": "1e-08",
        "14": "3e-07",
        "20": "2e+05",
        "21": "3e-02",
        "22": "1e-07",
        "23": "1e-08",
        "24": "3e-07",
        "30": "2e+05",
        "31": "3e-02",
        "32": "1e-07",
        "33": "1e-08",
        "34": "3e-07",
        "40": "2e+05",
        "41": "3e-02",
        "42": "1e-07",
        "43": "1e-08",
        "44": "3e-07"
    },
    "1": {
        "00": "2e+05",
        "01": "3e-02",
        "02": "1e-07",
        "03": "1e-08",
        "04": "3e-07",
        "10": "2e+05",
        "11": "3e-02",
        "12": "1e-07",
        "13": "1e-08",
        "14": "3e-07",
        "20": "2e+05",
        "21": "3e-02",
        "22": "1e-07",
        "23": "1e-08",
        "24": "3e-07",
        "30": "2e+05",
        "31": "3e-02",
    